In [ ]:
# Simplest Instance Segmentation with pretrained weights

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# 1. Load a pre-trained Mask R-CNN model from TensorFlow Hub
# Explore different Mask R-CNN models at https://tfhub.dev/s?q=mask+r-cnn
model_url = "https://tfhub.dev/tensorflow/mask_rcnn/resnet50_v1b_1024x1024/1"
detector = hub.load(model_url)

In [ ]:
# 2. Load and preprocess the image
image_path = "path/to/your/image.jpg"  # Replace with the path to your image
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
original_shape = img.shape[:2]

In [ ]:
# The model expects a batch of images
input_tensor = tf.convert_to_tensor(np.expand_dims(img, 0), dtype=tf.uint8)

In [ ]:
# 3. Perform instance segmentation
detections = detector(input_tensor)

In [ ]:
# 4. Extract detection information
detection_masks = detections['detection_masks'][0].numpy()
detection_boxes = detections['detection_boxes'][0].numpy()
detection_classes = detections['detection_classes'][0].numpy().astype(np.int32)
detection_scores = detections['detection_scores'][0].numpy()
num_detections = int(detections['num_detections'][0])

In [ ]:
# 5. Define class names (common for COCO dataset)
class_names = [
    'background', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag',
    'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite',
    'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana',
    'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table',
    'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock',
    'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

In [ ]:
# 6. Visualize the results
fig, ax = plt.subplots(1, figsize=(12, 12))
ax.imshow(img)

for i in range(num_detections):
    if detection_scores[i] > 0.5:  # Set a confidence threshold
        mask = detection_masks[i]
        bbox = detection_boxes[i]
        class_id = detection_classes[i]
        score = detection_scores[i]
        class_name = class_names[class_id]

        ymin, xmin, ymax, xmax = bbox
        (left, right, top, bottom) = (xmin * original_shape[1], xmax * original_shape[1],
                                      ymin * original_shape[0], ymax * original_shape[0])
        rect = patches.Rectangle((left, top), right - left, bottom - top,
                                 linewidth=1, edgecolor='r', facecolor='none')
        ax.add_patch(rect)

        mask_resized = cv2.resize(mask, (int(right - left), int(bottom - top)))
        bool_mask = mask_resized > 0.5
        masked_image = np.zeros_like(img, dtype=np.uint8)
        masked_image[int(top):int(bottom), int(left):int(right)][bool_mask] = [0, 255, 0]  # Green mask

        ax.imshow(masked_image, alpha=0.3)
        ax.text(left, top - 10, f"{class_name}: {score:.2f}", fontsize=8, color='r')

plt.axis('off')
plt.show()